In [1]:
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns

pd.set_option('display.max_columns', None)

In [2]:
df_age = pd.read_csv('ig_follower_demographics_age.csv', index_col=0)
df_city = pd.read_csv('ig_follower_demographics_city.csv', index_col=0)
df_country = pd.read_csv('ig_follower_demographics_country.csv', index_col=0)
df_gender = pd.read_csv('ig_follower_demographics_gender.csv', index_col=0)

df_city.sample(10)



,category,value
breakdown,,
city,"Seville, Andalusia",19
city,"Málaga, Andalusia",3
city,"Argentona, Cataluña",2
city,"Valladolid, Castilla y Leon",6
city,"Cáceres, Extremadura",2
city,"London, England",2
city,"Cullera, Comunidad Valenciana",2
city,"Murcia, Region of Murcia",10
city,"Pinto, Comunidad de Madrid",2


In [3]:
def unificar_demographics(dfs):
    
    dfs_limpios = []

    for df in dfs:

        print(df.columns.tolist())
        
        # solo reseteamos si hace falta
        if df.index.name == "breakdown":
            df = df.reset_index()
        # renombramos columnas
        df = df.rename(columns={
            "breakdown": "dimension",
            "value": "total_followers"
        })
        # comprobamos
        print(df.columns.tolist())
        # definimos el df con los nuevos nombres de columnas y añadimos a la lista vacía
        df = df[["dimension", "category", "total_followers"]]
        dfs_limpios.append(df)
    # unimos todos los datatasets a lo largo, unos debajo de otros
    df_unico = pd.concat(dfs_limpios, ignore_index=True)
    # renombramos los valores únicos de la columna dimension con el método .map
    map_dim = {
        "age": "edad",
        "gender": "género",
    }

    df_unico["dimension"] = df_unico["dimension"].map(map_dim)
    
    return df_unico
        
       

In [5]:
# definimos df_unico pasándole la función
dfs = [df_gender, df_age]
df_unico = unificar_demographics(dfs)

['category', 'value']
['dimension', 'category', 'total_followers']
['category', 'value']
['dimension', 'category', 'total_followers']


In [6]:
df_unico.sample()

,dimension,category,total_followers
7,edad,55-64,50


In [7]:
import pandas as pd
import numpy as np
import csv
import re

def export_to_tableau(
    df: pd.DataFrame,
    path: str,
    sep: str = ";",
    encoding: str = "utf-8"
):
    """
    Limpia y exporta un DataFrame a CSV de forma segura para Tableau.

    - Elimina saltos de línea en textos (evita filas rotas)
    - Normaliza nombres de columnas
    - Controla nulos (NaN / NaT)
    - Fuerza quoting completo
    - Usa separador europeo (;)

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame limpio en pandas
    path : str
        Ruta de salida del CSV
    sep : str, optional
        Separador del CSV (default ';')
    encoding : str, optional
        Encoding del archivo (default 'utf-8')
    """

    df = df.copy()

    # 1️⃣ Normalizar nombres de columnas
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(" ", "_")
          .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    # 2️⃣ Limpiar saltos de línea y retornos en columnas de texto
    cols_texto = df.select_dtypes(include="object").columns
    df[cols_texto] = df[cols_texto].replace(
        {r"[\r\n]+": " "},
        regex=True
    )

    # 3️⃣ Convertir NaN / NaT a None (Tableau-friendly)
    df = df.replace({pd.NA: None, np.nan: None})

    # 4️⃣ Exportar CSV robusto
    df.to_csv(
        path,
        sep=sep,
        index=False,
        encoding=encoding,
        quoting=csv.QUOTE_ALL
    )

    print(f"✅ CSV exportado correctamente para Tableau: {path}")
    print(f"📊 Filas: {df.shape[0]} | Columnas: {df.shape[1]}")

In [8]:
export_to_tableau(df_unico, 'demographics_IG.csv')

✅ CSV exportado correctamente para Tableau: demographics_IG.csv
📊 Filas: 9 | Columnas: 3
